# C6-pytorch — Practice p10 — Solution


The plane scores are $x_1$ and $2-x_1-x_2$.  After the shared gate,
both are bits; their sum reaches two exactly when both half-plane
conditions hold, so the combine score is $h_1+h_2-1.5$ (the pinned $-(2-0.5)$ AND bias).


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


class RegionDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.plane = DenseLayer(torch.tensor([[1.0, 0.0], [-1.0, -1.0]]),
                                torch.tensor([0.0, 2.0]))
        self.gate = ThresholdGate()
        self.combine = DenseLayer(torch.tensor([[1.0, 1.0]]),
                                  torch.tensor([-1.5]))   # AND of 2: -(2 - 0.5), the course's pinned half-integer bias

    def forward(self, x):
        half_planes = self.gate(self.plane(x))
        return self.gate(self.combine(half_planes))


torch.manual_seed(20260804)
pts = torch.rand(500, 2) * 4 - 2

det = RegionDetector()
param_names = sorted(name for name, _ in det.named_parameters())
verdict = det(pts).ravel()
n_hits = int(verdict.sum())
direct = ((pts[:, 0] >= 0) & (pts[:, 0] + pts[:, 1] <= 2)).to(pts.dtype)
frac_agree = float((verdict == direct).to(pts.dtype).mean())

param_names, n_hits, frac_agree


### Answer check


In [ ]:
assert param_names == ["combine.bias", "combine.weight", "plane.bias", "plane.weight"]
assert verdict.shape == (500,)
assert n_hits == 192
assert frac_agree == 1.0
assert not det.plane.weight.requires_grad and not det.plane.bias.requires_grad
assert not det.combine.weight.requires_grad and not det.combine.bias.requires_grad
